# Node metrics analysis - Full reports AAL89

Testing for significant differences in Degree distribution between healthy controls and comatose patients. fMRI and TSPO graphs. Only for graphs at 10% density, comparing healthy controls and comatose patients only (no subgroup analyses).

To repeat the analyses with a different parcellation, change the indicated paths.

In [ ]:
import numpy as np
import networkx as nx
import scipy.special as ss
from networkx import tree
import os
import glob

def adj_matrix_connected(corr_matrix,sparsity_value):
    """given the correlation matrix and the expected sparsity coefficient it can 
    happen that the corresponding thresholded matrix results in a disconnected graph
    here we force the graph to be fully connected by the computation of the minimum
    spanning tree and adding the required edges in order to have a unique connected component 
    """
    if sparsity_value == 1.0:
        adj_matrix=np.ones(corr_matrix.shape)
        np.fill_diagonal(adj_matrix,0)
        return adj_matrix
        
    
    corr_matrix =abs(corr_matrix)

    max_num_edges = ss.comb(corr_matrix.shape[0],2)
    num_edges = int(max_num_edges*sparsity_value)
    
    num_regions=corr_matrix.shape[0]
    #total number of regions in the graph
        
    totalgraph=nx.from_numpy_array(1-abs(corr_matrix))
    #extraction of a complete graph having has weight 1-abs(correlation)
    #we need to take 1-abs since the mst is taking the minimum weight graph and we want the most correlated edges to be there
    
    MST=nx.to_numpy_array(tree.minimum_spanning_tree(totalgraph).to_undirected())
    MST_adj_mat=MST
    MST_adj_mat[MST>0]==1
    MST_adj_mat=np.triu(MST_adj_mat) #put zeros in the inferior triangular matrix
    
    #put zeros in the diagonal of the corr matrix
    for i in range(num_regions):
        corr_matrix[i,i]=0
    
    values_corr=abs(np.triu(corr_matrix))
    
    cor_wo_MST=values_corr[np.triu(MST_adj_mat)==0]
    #we do not consider the correlation values which do not involve edges that are already in the MST
    
    values=list(cor_wo_MST.flatten())
    values.sort(reverse=True)
    
    #we select the maximum value of correlation to have the expected num of edges - num of edges in the mst (num regions-1)
    value_thresh=values[num_edges-(num_regions-1)-1] #-1 index start at 0
    
    adj_matrix=np.zeros(corr_matrix.shape) 
    
    #we put an edge if the value of correlation is higher than the found threshold or if the edges is required by the mst
    adj_matrix[values_corr>=value_thresh]=1
    adj_matrix[MST_adj_mat!=0]=1
    
    adj_matrix=np.triu(adj_matrix)+np.transpose(np.triu(adj_matrix)) #simmetry of the adj matrix
    
    return adj_matrix


fMRI data

In [ ]:
import os
import glob
controls_aal = [
    "01FO", "02LE", "03GA", "04GM", "05IM", "07NA", "08CP", "09DM", "11GL", "12LJ",
    "13AE", "14PM", "15GT", "16DT", "17LY", "19DG", "20CP", "21LJ", "22DD", "23BA"
]

anoxic_aal = [ # Effective Anoxic patients
    "01JF", "02PD", "06BM", "07TA", "14RC"
]
traumatic_aal = [ # Effective Traumatic patients- Patient 08PE must be excluded from studies using AICHA 
    "03DB", "08PE", "11FC", "13TL", "16FF", "22BT", "23GC", "24ZX", "26AC"
]

matrices_path = "/Data/Correlation_Matrices/AAL" # Simply change the last path to import other atlases, e.g. AICHA
control_path = os.path.join(matrices_path, "Controls")
anoxic_path = os.path.join(matrices_path, "Anoxic")
traumatic_path = os.path.join(matrices_path, "Traumatic")

# Final filter before importing - matching IDs from lists to files to exclude non-effective subjects

control_subjects = sorted(
    s for s in glob.glob(os.path.join(control_path, "*"))
    if os.path.basename(s).split(".")[0] in controls_aal
)

anoxic_subjects = sorted(
    s for s in glob.glob(os.path.join(anoxic_path, "*"))
    if os.path.basename(s).split(".")[0] in anoxic_aal
)

traumatic_subjects = sorted(
    s for s in glob.glob(os.path.join(traumatic_path, "*"))
    if os.path.basename(s).split(".")[0] in traumatic_aal
)

# Importing correlations - Keys are IDs, values are matrices

control_correlations = {}
for sub in control_subjects:
    control_correlations[os.path.splitext(os.path.basename(sub))[0]] = np.loadtxt(sub)


traumatic_correlations = {}
for sub in traumatic_subjects:
    traumatic_correlations[os.path.splitext(os.path.basename(sub))[0]] = np.loadtxt(sub)

anoxic_correlations = {}
for sub in anoxic_subjects:
    anoxic_correlations[os.path.splitext(os.path.basename(sub))[0]] = np.loadtxt(sub)

print(f"Controls: {len(control_correlations.keys())}, Anoxic: {len(anoxic_correlations.keys())}, Traumatic: {len(traumatic_correlations.keys())}")

# Constructing fMRI graphs

costs = [0.1] # costs: 5%, 10%, and so on
controls_fmri_graphs = {cost:{sub: None for sub in control_correlations.keys()} for cost in costs}
for cost in costs:
    for sub in control_correlations.keys(): # Keys are IDs, values are nx.Graph objects
        controls_fmri_graphs[cost][sub] = nx.from_numpy_array(adj_matrix_connected(control_correlations[sub], cost))

anoxic_fmri_graphs = {cost:{sub: None for sub in anoxic_correlations.keys()} for cost in costs}
for cost in costs:
    for sub in anoxic_correlations.keys():
        anoxic_fmri_graphs[cost][sub] = nx.from_numpy_array(adj_matrix_connected(anoxic_correlations[sub], cost))

traumatic_fmri_graphs = {cost:{sub: None for sub in traumatic_correlations.keys()} for cost in costs}
for cost in costs:
    for sub in traumatic_correlations.keys():
        traumatic_fmri_graphs[cost][sub] = nx.from_numpy_array(adj_matrix_connected(traumatic_correlations[sub], cost))

# We can merge the coma dataset in one dictionary

coma_fmri_graphs = {cost: {} for cost in costs}

for cost in costs:
    coma_fmri_graphs[cost] = {**anoxic_fmri_graphs[cost], **traumatic_fmri_graphs[cost]}


Controls: 20, Anoxic: 5, Traumatic: 9


In [3]:
favorable_group = ["06BM", "11FC", "13TL", "14RC", "22BT", "23GC", "24ZX", "25AY", "26AC"]
unfavorable_group = ["01JF", "02PD", "03DB", "07TA", "08PE", "09CD", "16FF", "20MP"]

PET data

In [ ]:
tspo_matrix_path = "/Data/PET_Matrices/AAL" # Simply change the last path to import other atlases, e.g. AICHA

########### Controls
list_tspo_cn = os.listdir(os.path.join(tspo_matrix_path, "Controls"))

controls_tspo_matrices = {}

for f in list_tspo_cn:
    #sub = 
    sub = f.split(".")[0]
    controls_tspo_matrices[sub] = np.loadtxt(os.path.join(tspo_matrix_path, "Controls", f), delimiter=",")

list_tspo_anox = os.listdir(os.path.join(tspo_matrix_path, "Anoxic"))


########### Anoxic
# --- Initialize container ---
anoxic_tspo_matrices = {}

# --- Load matrices, excluding unwanted IDs ---
for f in list_tspo_anox:
    sub = f.split(".")[0]  # extract subject ID (before .csv or .txt)
    
    if sub not in anoxic_aal:
        print(f"Skipping excluded subject: {sub}")
        continue  # skip loading this subject
    
    file_path = os.path.join(tspo_matrix_path, "Anoxic", f)
    anoxic_tspo_matrices[sub] = np.loadtxt(file_path, delimiter=",")


########### Traumatic

list_tspo_trau = os.listdir(os.path.join(tspo_matrix_path, "Traumatic"))

# --- Initialize container ---
traumatic_tspo_matrices = {}

# --- Load matrices, excluding unwanted IDs ---
for f in list_tspo_trau:
    sub = f.split(".")[0]  # extract subject ID (before .csv or .txt)
    
    if sub not in traumatic_aal:
        print(f"Skipping excluded subject: {sub}")
        continue  # skip loading this subject
    
    file_path = os.path.join(tspo_matrix_path, "Traumatic", f)
    traumatic_tspo_matrices[sub] = np.loadtxt(file_path, delimiter=",")

t_costs = [0.1] #[0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]

controls_tspo_graphs = {cost:{sub: None for sub in controls_tspo_matrices.keys()} for cost in t_costs}
for cost in t_costs:
    for sub in controls_tspo_matrices.keys():
        controls_tspo_graphs[cost][sub] = nx.from_numpy_array(adj_matrix_connected(controls_tspo_matrices[sub], cost))

anoxic_tspo_graphs = {cost:{sub: None for sub in anoxic_tspo_matrices.keys()} for cost in t_costs}
for cost in t_costs:
    for sub in anoxic_tspo_matrices.keys():
            anoxic_tspo_graphs[cost][sub] = nx.from_numpy_array(adj_matrix_connected(anoxic_tspo_matrices[sub], cost))

traumatic_tspo_graphs = {cost:{sub: None for sub in traumatic_tspo_matrices.keys()} for cost in t_costs}
for cost in t_costs:
    for sub in traumatic_tspo_matrices.keys():
        traumatic_tspo_graphs[cost][sub] = nx.from_numpy_array(adj_matrix_connected(traumatic_tspo_matrices[sub], cost))

# We can merge the coma dataset in one dictionary

coma_tspo_graphs = {cost: {} for cost in costs}

for cost in costs:
    coma_tspo_graphs[cost] = {**anoxic_tspo_graphs[cost], **traumatic_tspo_graphs[cost]}


Skipping excluded subject: 09CD
Skipping excluded subject: 20MP
Skipping excluded subject: 25AY


In [5]:
allowed_favorable = set(coma_tspo_graphs[0.1].keys()).intersection(set(favorable_group))
allowed_unfavorable = set(coma_tspo_graphs[0.1].keys()).intersection(set(unfavorable_group))

code, refactored

In [ ]:
import sys
import numpy as np
import pandas as pd
from collections import defaultdict
import networkx as nx
from scipy.stats import mannwhitneyu

sys.path.append("/source/multipy-master") # or pip install multipy-master
from multipy.fwer import bonferroni
from multipy.fdr import lsu

# ─────────────────────────────────────────────
# GRAPH METRICS
# ─────────────────────────────────────────────

def compute_graph_metrics(G):
    """Compute per-node graph metrics for a NetworkX graph."""
    return {
        "d":  dict(G.degree()),
        "cc": nx.clustering(G),
        "cs": nx.closeness_centrality(G),
    }

def populate_group_metrics(subj_graphs, group_name, group_metrics,
                           favorable_set=None, unfavorable_set=None):
    """
    For each subject graph, compute metrics and accumulate them per node into
    group_metrics[group_name][node][metric] lists.

    Coma     = union of Anoxic + Traumatic.
    Favorable / Unfavorable are filled when outcome sets are provided.
    """
    for sub, G in subj_graphs.items():
        metrics   = compute_graph_metrics(G)
        int_nodes = [n for n in G.nodes if isinstance(n, int)]

        def store(target_group):
            for node in int_nodes:
                for metric_name, metric_dict in metrics.items():
                    group_metrics[target_group][node][metric_name].append(metric_dict[node])

        store(group_name)

        if group_name in ("Anoxic", "Traumatic"):
            store("Coma")

        if favorable_set is not None and sub in favorable_set:
            store("Favorable")
        if unfavorable_set is not None and sub in unfavorable_set:
            store("Unfavorable")

    # Remove any accidentally inserted non-integer keys
    for grp in group_metrics:
        for k in [k for k in group_metrics[grp] if not isinstance(k, int)]:
            del group_metrics[grp][k]


# ─────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────

COSTS = [0.1]
GROUPS    = ["Controls", "Anoxic", "Traumatic", "Coma", "Favorable", "Unfavorable"]
METRICS   = ["d"] 
N_REGIONS = 89 # Change depending on the number of regions of the chosen atlas

GROUP_PAIRS = [("Controls",  "Coma")]

# Load region names (done once, shared across all costs)
region_names = (
    pd.read_excel("/Atlases/AALours/List_regions.xlsx") # Change depending on the used atlas
    .iloc[:, 0]
    .tolist()
)


# ─────────────────────────────────────────────
# MAIN LOOP OVER COSTS
# all_tables[metric][cost][pair_label] = DataFrame
# ─────────────────────────────────────────────

all_tables = {metric: {} for metric in METRICS}

for cost in COSTS:
    print(f"\n{'#'*90}")
    print(f"  COST: {cost}")
    print(f"{'#'*90}")

    # ── 1. Populate group metrics for this cost ──────────────────────────────

    group_metrics = {g: defaultdict(lambda: defaultdict(list)) for g in GROUPS}

    populate_group_metrics(controls_fmri_graphs[cost],  "Controls",  group_metrics)
    populate_group_metrics(anoxic_fmri_graphs[cost],    "Anoxic",    group_metrics,
                           favorable_set=allowed_favorable, unfavorable_set=allowed_unfavorable)
    populate_group_metrics(traumatic_fmri_graphs[cost], "Traumatic", group_metrics,
                           favorable_set=allowed_favorable, unfavorable_set=allowed_unfavorable)

    # ── 2. Mann-Whitney U tests — (metric, pair, region) ────────────────────

    # pval_matrix[metric]  → array (N_REGIONS × N_PAIRS)
    # ustat_matrix[metric] → array (N_REGIONS × N_PAIRS)
    pval_matrix  = {m: np.full((N_REGIONS, len(GROUP_PAIRS)), np.nan) for m in METRICS}
    ustat_matrix = {m: np.full((N_REGIONS, len(GROUP_PAIRS)), np.nan) for m in METRICS}

    for metric in METRICS:
        for pair_idx, (group_a, group_b) in enumerate(GROUP_PAIRS):
            for region in range(N_REGIONS):
                data_a = group_metrics[group_a][region][metric]
                data_b = group_metrics[group_b][region][metric]

                if len(data_a) > 1 and len(data_b) > 1:
                    u, p = mannwhitneyu(data_a, data_b, alternative="two-sided")
                else:
                    u, p = np.nan, np.nan

                pval_matrix[metric][region, pair_idx]  = p
                ustat_matrix[metric][region, pair_idx] = u

    # ── 3. Multiple comparisons correction — across N_REGIONS per (metric, pair) ──

    correction = {}

    for metric in METRICS:
        print(f"\n=== Correction — metric: {metric} | cost: {cost} ===")
        correction[metric] = {}

        for pair_idx, (group_a, group_b) in enumerate(GROUP_PAIRS):
            pair_label = f"{group_a}_{group_b}"
            pvals_pair = pval_matrix[metric][:, pair_idx]  # length N_REGIONS

            rej_fdr  = lsu(pvals_pair, q=0.1)

            correction[metric][pair_label] = {"fdr": rej_fdr}

            print(f"  {pair_label:35s} | FDR: {np.sum(rej_fdr):3d}")

    # ── 4. Build full results tables and store in all_tables ─────────────────

    for metric in METRICS:
        # Initialise cost-level dict on first encounter
        if cost not in all_tables[metric]:
            all_tables[metric][cost] = {}

        for pair_idx, (group_a, group_b) in enumerate(GROUP_PAIRS):
            pair_label = f"{group_a}_{group_b}"
            fdr_mask   = correction[metric][pair_label]["fdr"]

            rows = []
            for region in range(N_REGIONS):
                data_a = group_metrics[group_a][region][metric]
                data_b = group_metrics[group_b][region][metric]

                mean_a = np.mean(data_a) if data_a else np.nan
                mean_b = np.mean(data_b) if data_b else np.nan


                rows.append({
                    "region_idx":      region,
                    "region_name":     region_names[region],
                    f"mean_{group_a}": round(mean_a, 4),
                    f"mean_{group_b}": round(mean_b, 4),
                    "U_statistic":     ustat_matrix[metric][region, pair_idx],
                    "p_value":         pval_matrix[metric][region, pair_idx],
                    "fdr_sig":         bool(fdr_mask[region]),
                })

            all_tables[metric][cost][pair_label] = pd.DataFrame(rows)

# ── Example access ───────────────────────────────────────────────────────────
# all_tables["d"][0.05]["Controls_Anoxic"]
# all_tables["cc"][0.10]["Favorable_Unfavorable"]


##########################################################################################
  COST: 0.1
##########################################################################################

=== Correction — metric: d | cost: 0.1 ===
  Controls_Coma                       | FDR:  24


In [7]:
all_tables["d"][0.1]['Controls_Coma'].loc[all_tables["d"][0.1]['Controls_Coma']['fdr_sig'] == True]

,region_idx,region_name,mean_Controls,mean_Coma,U_statistic,p_value,fdr_sig
5,5,FrontSupOrb_R,2.05,7.0000,72.5,0.008582,True
6,6,FrontMid_L,12.20,18.5714,71.5,0.017089,True
7,7,FrontMid_R,11.95,20.6429,59.5,0.005002,True
8,8,FrontMidOrb_L,3.30,6.9286,70.0,0.013386,True
9,9,FrontMidOrb_R,3.10,8.0000,63.0,0.006800,True
12,12,FrontInfTri_L,5.35,12.0000,48.0,0.001292,True
13,13,FrontInfTri_R,4.80,13.1429,48.5,0.001366,True
24,24,FrontMedOrb_L,3.25,6.5000,64.0,0.007208,True
30,30,CingMid_L,16.05,6.1429,256.0,0.000052,True
31,31,CingMid_R,14.15,9.1429,205.0,0.023598,True


In [ ]:
# ─────────────────────────────────────────────
# EXPORT — one CSV per (cost, pair)
# Structure: results/{cost}_{pair}.csv
# ─────────────────────────────────────────────

BASE_OUTPUT_DIR = "/D/AAL"  # adjust as needed
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

for metric, costs_dict in all_tables.items():
    for cost, pairs_dict in costs_dict.items():

        # Format cost: 0.05 → "005"
        cost_str = str(cost).replace('.', '')

        for pair_label, df in pairs_dict.items():
            filename = f"{cost_str}_{pair_label}_fMRI_Degree.csv"
            out_path = os.path.join(BASE_OUTPUT_DIR, filename)

            df.to_csv(out_path, index=False)
            # print(f"Saved: {out_path}")

We repeat the same process for TSPO

In [ ]:
# ─────────────────────────────────────────────
# MAIN LOOP OVER COSTS
# all_tables_tspo[metric][cost][pair_label] = DataFrame
# ─────────────────────────────────────────────

all_tables_tspo = {metric: {} for metric in METRICS}

for cost in COSTS:
    print(f"\n{'#'*90}")
    print(f"  COST: {cost}")
    print(f"{'#'*90}")

    # ── 1. Populate group metrics for this cost ──────────────────────────────

    group_metrics = {g: defaultdict(lambda: defaultdict(list)) for g in GROUPS}

    populate_group_metrics(controls_tspo_graphs[cost],  "Controls",  group_metrics)
    populate_group_metrics(anoxic_tspo_graphs[cost],    "Anoxic",    group_metrics,
                           favorable_set=allowed_favorable, unfavorable_set=allowed_unfavorable)
    populate_group_metrics(traumatic_tspo_graphs[cost], "Traumatic", group_metrics,
                           favorable_set=allowed_favorable, unfavorable_set=allowed_unfavorable)

    # ── 2. Mann-Whitney U tests — (metric, pair, region) ────────────────────

    # pval_matrix[metric]  → array (N_REGIONS × N_PAIRS)
    # ustat_matrix[metric] → array (N_REGIONS × N_PAIRS)
    pval_matrix  = {m: np.full((N_REGIONS, len(GROUP_PAIRS)), np.nan) for m in METRICS}
    ustat_matrix = {m: np.full((N_REGIONS, len(GROUP_PAIRS)), np.nan) for m in METRICS}

    for metric in METRICS:
        for pair_idx, (group_a, group_b) in enumerate(GROUP_PAIRS):
            for region in range(N_REGIONS):
                data_a = group_metrics[group_a][region][metric]
                data_b = group_metrics[group_b][region][metric]

                if len(data_a) > 1 and len(data_b) > 1:
                    u, p = mannwhitneyu(data_a, data_b, alternative="two-sided")
                else:
                    u, p = np.nan, np.nan

                pval_matrix[metric][region, pair_idx]  = p
                ustat_matrix[metric][region, pair_idx] = u

    # ── 3. Multiple comparisons correction — across N_REGIONS per (metric, pair) ──

    correction = {}

    for metric in METRICS:
        print(f"\n=== Correction — metric: {metric} | cost: {cost} ===")
        correction[metric] = {}

        for pair_idx, (group_a, group_b) in enumerate(GROUP_PAIRS):
            pair_label = f"{group_a}_{group_b}"
            pvals_pair = pval_matrix[metric][:, pair_idx]  # length N_REGIONS

            rej_fdr  = lsu(pvals_pair, q=0.1)

            correction[metric][pair_label] = {"fdr": rej_fdr}

            print(f"  {pair_label:35s} | FDR: {np.sum(rej_fdr):3d}")

    # ── 4. Build full results tables and store in all_tables_tspo ─────────────────

    for metric in METRICS:
        # Initialise cost-level dict on first encounter
        if cost not in all_tables_tspo[metric]:
            all_tables_tspo[metric][cost] = {}

        for pair_idx, (group_a, group_b) in enumerate(GROUP_PAIRS):
            pair_label = f"{group_a}_{group_b}"
            fdr_mask   = correction[metric][pair_label]["fdr"]

            rows = []
            for region in range(N_REGIONS):
                data_a = group_metrics[group_a][region][metric]
                data_b = group_metrics[group_b][region][metric]

                mean_a = np.mean(data_a) if data_a else np.nan
                mean_b = np.mean(data_b) if data_b else np.nan

                rows.append({
                    "region_idx":      region,
                    "region_name":     region_names[region],
                    f"mean_{group_a}": round(mean_a, 4),
                    f"mean_{group_b}": round(mean_b, 4),
                    "U_statistic":     ustat_matrix[metric][region, pair_idx],
                    "p_value":         pval_matrix[metric][region, pair_idx],
                    "fdr_sig":         bool(fdr_mask[region]),
                })

            all_tables_tspo[metric][cost][pair_label] = pd.DataFrame(rows)


##########################################################################################
  COST: 0.1
##########################################################################################

=== Correction — metric: d | cost: 0.1 ===
  Controls_Coma                       | FDR:   1


In [9]:
all_tables_tspo["d"][0.1]['Controls_Coma'].loc[all_tables_tspo["d"][0.1]['Controls_Coma']['fdr_sig'] == True]

,region_idx,region_name,mean_Controls,mean_Coma,U_statistic,p_value,fdr_sig
88,88,Vermis,3.25,10.1429,31.5,0.000124,True


In [ ]:
# ─────────────────────────────────────────────
# EXPORT — one CSV per (cost, pair)
# Structure: results/{cost}_{pair}.csv
# ─────────────────────────────────────────────

BASE_OUTPUT_DIR = "/D/AAL"  # adjust as needed
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

for metric, costs_dict in all_tables_tspo.items():
    for cost, pairs_dict in costs_dict.items():

        # Format cost: 0.05 → "005"
        cost_str = str(cost).replace('.', '')

        for pair_label, df in pairs_dict.items():
            filename = f"{cost_str}_{pair_label}_TSPO_Degree.csv"
            out_path = os.path.join(BASE_OUTPUT_DIR, filename)

            df.to_csv(out_path, index=False)
            # print(f"Saved: {out_path}")